# Stage 1 Detection Ensemble — Colab Training

Trains the paper's 4-model detection ensemble (Normal vs Pneumonia) on Google Colab:
- **PneumoNet-v1** (custom CNN, paper weight 1)
- **PneumoNet-v2** (custom CNN, paper weight 3)
- **VGG-16** (transfer, paper weight 5)
- **Inception-v3** (transfer, paper weight 4)

Paper target: **98.03% accuracy, AUC 0.97** (published weights VGG-16:5, Inception-v3:4, PneumoNet-v2:3, PneumoNet-v1:1).

**Before running:** Runtime → Change runtime type → **T4 GPU**.
All checkpoints save to Google Drive so nothing is lost if the session disconnects.

## 1. Mount Google Drive (persistence for checkpoints & data)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DRIVE_DIR = '/content/drive/MyDrive/PneumoniaProject'
os.makedirs(PROJECT_DRIVE_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DRIVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_DRIVE_DIR}/data', exist_ok=True)
print('Drive project dir:', PROJECT_DRIVE_DIR)

## 2. Clone the GitHub repo

In [ ]:
# If the repo is private, uncomment and provide a GitHub Personal Access Token:
# import getpass
# token = getpass.getpass('GitHub PAT: ')
# REPO_URL = f'https://{token}@github.com/peas-rip/IEEE-Projects.git'

REPO_URL = 'https://github.com/peas-rip/IEEE-Projects.git'
import os
if not os.path.exists('/content/IEEE-Projects'):
    !git clone {REPO_URL} /content/IEEE-Projects

%cd /content/IEEE-Projects
!ls

**Note:** the repo root is `IEEE-Projects`, and our project lives in the `ML Projects/Pnemonia Treatment` subfolder. Point the shell at it:

In [ ]:
PROJECT_DIR = '/content/IEEE-Projects/ML Projects/Pnemonia Treatment'
import os
os.chdir(PROJECT_DIR)
os.environ['PYTHONPATH'] = os.path.join(PROJECT_DIR, 'src')
print('Working dir:', os.getcwd())
!ls src/pneumonia configs

## 3. Install dependencies (torch is preinstalled on Colab)

In [ ]:
!pip install -q scikit-learn pyyaml pillow kagglehub numpy

import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (switch runtime to T4 GPU!)')

## 4. Kaggle dataset via kagglehub

Upload your `kaggle.json` API token (Kaggle → Account → Create New Token) when prompted.

In [ ]:
import os

# One-time setup: upload kaggle.json
if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
    from google.colab import files
    print('Upload your kaggle.json (Kaggle → Account → Create New Token)')
    uploaded = files.upload()
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    os.rename('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

print('Kaggle credentials ready.')

In [ ]:
import kagglehub

path = kagglehub.dataset_download('paultimothymooney/chest-xray-pneumonia')
print('Dataset downloaded to:', path)

# Locate the chest_xray folder
import os
for root, dirs, files in os.walk(path):
    if 'chest_xray' in dirs:
        DATASET_ROOT = os.path.join(root, 'chest_xray')
        break

print('Dataset root:', DATASET_ROOT)
for split in ['train', 'test', 'val']:
    split_dir = os.path.join(DATASET_ROOT, split)
    if os.path.isdir(split_dir):
        print(f'{split}:', {c: len(os.listdir(os.path.join(split_dir, c))) for c in os.listdir(split_dir)})

## 5. Reorganize data (Stage 1: NORMAL/PNEUMONIA + Stage 2: BACTERIAL/VIRAL)
Saved to Drive so we never re-download/reorganize across sessions.

In [ ]:
import os, shutil

base = DATASET_ROOT
out_stage1 = f'{PROJECT_DRIVE_DIR}/data/stage1'   # NORMAL / PNEUMONIA
out_stage2 = f'{PROJECT_DRIVE_DIR}/data/stage2'   # BACTERIAL / VIRAL

def classify(fname):
    f = fname.lower()
    if 'bacteria' in f: return 'BACTERIAL'
    if 'virus' in f: return 'VIRAL'
    return None

if not os.path.exists(f'{out_stage1}/train/NORMAL'):
    for split in ['train', 'test']:
        src = os.path.join(base, split)
        # Stage 1: NORMAL vs PNEUMONIA (same as original layout)
        for cls in ['NORMAL', 'PNEUMONIA']:
            os.makedirs(f'{out_stage1}/{split}/{cls}', exist_ok=True)
            for f in os.listdir(f'{src}/{cls}'):
                if not f.startswith('.'):  # skip __MACOSX junk
                    shutil.copy2(f'{src}/{cls}/{f}', f'{out_stage1}/{split}/{cls}/{f}')
        # Stage 2: BACTERIAL vs VIRAL (from PNEUMONIA filenames)
        for cls in ['BACTERIAL', 'VIRAL']:
            os.makedirs(f'{out_stage2}/{split}/{cls}', exist_ok=True)
        for f in os.listdir(f'{src}/PNEUMONIA'):
            c = classify(f)
            if c:
                shutil.copy2(f'{src}/PNEUMONIA/{f}', f'{out_stage2}/{split}/{c}/{f}')
    print('Data reorganized to Drive.')
else:
    print('Stage 1 data already on Drive - skipping.')

print('\nStage 1 (NORMAL vs PNEUMONIA):')
for split in ['train', 'test']:
    print(f'  {split}:', {c: len(os.listdir(f'{out_stage1}/{split}/{c}')) for c in ['NORMAL', 'PNEUMONIA']})
print('Stage 2 (BACTERIAL vs VIRAL):')
for split in ['train', 'test']:
    print(f'  {split}:', {c: len(os.listdir(f'{out_stage2}/{split}/{c}')) for c in ['BACTERIAL', 'VIRAL']})

## 6. Sanity check: quick training run on a tiny subset
Validates data paths, GPU and class weighting before the long runs (~1-2 min).

In [ ]:
import os, shutil, random
random.seed(42)

# Build a tiny subset (~80 imgs/class from train) for the sanity run
SANITY_DIR = '/content/sanity_data'
if not os.path.exists(f'{SANITY_DIR}/NORMAL'):
    STAGE1_TRAIN = f'{PROJECT_DRIVE_DIR}/data/stage1/train'
    for cls in ['NORMAL', 'PNEUMONIA']:
        os.makedirs(f'{SANITY_DIR}/{cls}', exist_ok=True)
        files = os.listdir(f'{STAGE1_TRAIN}/{cls}')
        for f in random.sample(files, 80):
            shutil.copy2(f'{STAGE1_TRAIN}/{cls}/{f}', f'{SANITY_DIR}/{cls}/{f}')

# Config has epochs=15, but on 160 small images a full run takes ~1-2 min
!python -m pneumonia train \
    --config configs/stage1_pneumonet_v1.yaml \
    --data "{SANITY_DIR}" \
    --checkpoint-dir /content/sanity_check

print('\nSanity run passed. Data paths, GPU and class weighting all work.')

## 7. Train the 4 Stage 1 models (sequential, checkpoints to Drive)
Each model trains on the full Stage 1 train set (~15.6k images after 80/20 split). Estimated: ~10-20 min/epoch on T4 depending on model. Checkpoints save to Drive every epoch.

**Tip:** run these cells one at a time. If a session disconnects, re-run cell 1 (Drive mount) + cell 2 paths, then jump straight to the next untrained model.

In [ ]:
import os

STAGE1_DATA = f'{PROJECT_DRIVE_DIR}/data/stage1/train'
CKPT_BASE = f'{PROJECT_DRIVE_DIR}/checkpoints'

models_to_train = [
    ('stage1_pneumonet_v1', 'pneumonet_v1'),
    ('stage1_pneumonet_v2', 'pneumonet_v2'),
    ('stage1_vgg16',        'vgg16'),
    ('stage1_inception',    'inception_v3'),
]

for cfg_name, model_name in models_to_train:
    ckpt_dir = f'{CKPT_BASE}/{cfg_name}'
    if os.path.exists(f'{ckpt_dir}/final_model.pt'):
        print(f'[SKIP] {model_name} already trained: {ckpt_dir}/final_model.pt exists')
        continue
    print(f'\n{"="*60}\nTraining {model_name} (config: configs/{cfg_name}.yaml)\n{"="*60}')
    !python -m pneumonia train \
        --config configs/{cfg_name}.yaml \
        --data "{STAGE1_DATA}" \
        --checkpoint-dir "{ckpt_dir}"

**Fallback: single-model training cell** — if the loop cell has trouble, run each model individually with: 

In [ ]:
# Run any single model like this (edit the config name):
CFG = 'stage1_pneumonet_v1'         # stage1_pneumonet_v1 | stage1_pneumonet_v2 | stage1_vgg16 | stage1_inception
STAGE1_DATA = f'{PROJECT_DRIVE_DIR}/data/stage1/train'
CKPT_DIR = f'{PROJECT_DRIVE_DIR}/checkpoints/{CFG}'

!python -m pneumonia train \
    --config configs/{CFG}.yaml \
    --data "{STAGE1_DATA}" \
    --checkpoint-dir "{CKPT_DIR}"

## 8. Evaluate each model individually (before the ensemble)

In [ ]:
STAGE1_TEST = f'{PROJECT_DRIVE_DIR}/data/stage1/test'
CKPT_BASE = f'{PROJECT_DRIVE_DIR}/checkpoints'

# NOTE: evaluate now runs on the FULL directory passed via --data (all test images)
for cfg_name in ['stage1_pneumonet_v1', 'stage1_pneumonet_v2', 'stage1_vgg16', 'stage1_inception']:
    print(f'\n{"="*60}\nEvaluating {cfg_name}\n{"="*60}')
    !python -m pneumonia evaluate \
        --config configs/{cfg_name}.yaml \
        --data "{STAGE1_TEST}" \
        --checkpoint "{CKPT_BASE}/{cfg_name}/final_model.pt"

## 9. Evaluate the full Stage 1 ensemble (paper weights: VGG-16:5, Inception:4, PneumoNet-v2:3, PneumoNet-v1:1)

The script evaluates on the test split derived from `--data`. For Stage 1 ensemble we point it at the Stage 1 data root; it computes NORMAL vs PNEUMONIA binary metrics.

In [ ]:
# Point the ensemble script at the Drive checkpoints + Stage 1 test set
import re

script_path = 'scripts/evaluate_stage1_ensemble.py'
with open(script_path) as f:
    src = f.read()
src = src.replace('results/stage1_pneumonet_v1/final_model.pt', f'{PROJECT_DRIVE_DIR}/checkpoints/stage1_pneumonet_v1/final_model.pt')
src = src.replace('results/stage1_pneumonet_v2/final_model.pt', f'{PROJECT_DRIVE_DIR}/checkpoints/stage1_pneumonet_v2/final_model.pt')
src = src.replace('results/stage1_vgg16/final_model.pt', f'{PROJECT_DRIVE_DIR}/checkpoints/stage1_vgg16/final_model.pt')
src = src.replace('results/stage1_inception/final_model.pt', f'{PROJECT_DRIVE_DIR}/checkpoints/stage1_inception/final_model.pt')
src = src.replace('data/real_3class/test', f'{PROJECT_DRIVE_DIR}/data/stage1/test')
with open(script_path, 'w') as f:
    f.write(src)
print('Ensemble script paths updated to Drive.')

!python scripts/evaluate_stage1_ensemble.py

**Note:** `evaluate_stage1_ensemble.py` was written for the 3-class test layout (mapping BACTERIAL/VIRAL→PNEUMONIA). Pointing it at `stage1/test` (which has only NORMAL/PNEUMONIA classes) also works: the mapping just leaves NORMAL as 0 and labels PNEUMONIA as 1.

## 10. Save evaluation results to Drive

In [ ]:
import os
os.makedirs(f'{PROJECT_DRIVE_DIR}/results', exist_ok=True)

# Copy the ensemble script output into a results log (re-run to capture)
# For now, list all artifacts saved:
print('Checkpoints on Drive:')
for root, dirs, files in os.walk(f'{PROJECT_DRIVE_DIR}/checkpoints'):
    for f in files:
        if f == 'final_model.pt':
            path = os.path.join(root, f)
            size_mb = os.path.getsize(path) / 1e6
            print(f'  {path}  ({size_mb:.0f} MB)')

print('\nAll Stage 1 artifacts persisted to:', PROJECT_DRIVE_DIR)

## Checklist before training overnight / long runs
- [ ] Runtime = T4 GPU (16GB VRAM)
- [ ] Drive mounted (cell 1)
- [ ] kaggle.json uploaded (cell 4)
- [ ] Data reorganized on Drive (cell 5)
- [ ] Sanity 1-epoch run passed (cell 6)

**If the session disconnects mid-training:** epoch checkpoints are on Drive. Re-mount (cell 1), re-set paths (cell 2), then re-run the training cell for the affected model. Epoch checkpoints are stored as `checkpoint_epochN.pt`, and `final_model.pt` appears after the full run completes. To resume from a mid-training epoch instead of restarting, load `checkpoint_epochN.pt` and continue (the CLI currently restarts; resume support can be added if needed).